# Orynth — Open-Source Contributor Guide with Gemma 4
### HacktoberFest HackDay Theme 2B: RAG Application with Gemma

**Orynth** helps first-time open-source contributors understand repository documentation and take their first steps using answers grounded in real GitHub files, with source citations, visible retrieved passages, a safe refusal guarantee, and an onboarding checklist.

```
 GitHub Docs ──► 1. Split into chunks ──► 2. Embed with all-MiniLM-L6-v2 (CPU)
                                                       │
 Contributor Q ──► 3. Cosine similarity retrieval (Top-k: 3)
                                                       │
              4. Strict Gemma 4 prompt ──► Grounded Answer + File Citations
                 (If unsupported ──► Safe Refusal: "The answer could not be found")
```

**Models Used:**
- `sentence-transformers/all-MiniLM-L6-v2`: Runs locally in this notebook on CPU (no GPU required).
- `gemma-4-26b-a4b-it`: Google's open-weight model accessed via the free Google AI Studio API.

### Google AI Studio API Key Setup
1. Get your free key at **https://aistudio.google.com/apikey**
2. In Colab, open **Secrets** (key icon in left sidebar) and add `GOOGLE_API_KEY`.
*Note: In demonstration mode without an API key, the notebook runs full retrieval, citations, and transparent evidence extraction.*

## Step 0: Installation

In [ ]:
!pip install -q sentence-transformers google-genai "gradio>=6,<7" httpx

## Step 1: Configuration & Settings

In [ ]:
import os
import re
import time
import numpy as np

# Theme 2 required settings
GEMMA_MODEL = "gemma-4-26b-a4b-it"
CHUNK_SIZE = 120        # words per chunk
CHUNK_OVERLAP = 30      # word overlap between consecutive chunks
TOP_K = 3               # retrieved chunks given to Gemma
TEMPERATURE = 0.2       # low temperature ensures factual consistency
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MIN_RELEVANCE_THRESHOLD = 0.32

ANSWER_RULES = """You are Orynth, an evidence-grounded open-source contributor guide.
Answer questions about the repository using ONLY the context passages provided.
- Each passage begins with its file path and lines, e.g. [Passage 1: CONTRIBUTING.md (Lines 42-65)].
- For each fact you use, cite the file name in parentheses, e.g. (CONTRIBUTING.md).
- If the context does not contain the answer, reply exactly: "The answer could not be found in the repository documentation."
- Keep answers concise: 2 to 4 sentences."""

print("Settings configured successfully.")

## Step 2: Documentation Ingestion & Bundled Dataset

In [ ]:
# Bundled documentation dataset from pallets/flask for offline demonstration
SAMPLE_DOCS = {
    "README.md": """# Flask\n\nFlask is a lightweight WSGI web application framework. It is designed to make getting started quick and easy.\n\n## Installing\n\nInstall using pip:\n$ pip install -U Flask\n\nFlask requires Python 3.8 or newer.\n\n## Contributing\n\nFor guidance on setting up a development environment, see CONTRIBUTING.md.""",
    "CONTRIBUTING.md": """# How to Contribute to Flask\n\n## Setting Up a Development Sandbox\n\n1. Fork and clone the repository:\n   $ git clone https://github.com/your-username/flask.git\n2. Create a virtual environment:\n   $ python3 -m venv .venv\n   $ source .venv/bin/activate\n3. Install Flask in editable mode:\n   $ pip install -e \".[dev]\"\n4. Install pre-commit linters:\n   $ pre-commit install\n\n## Running Tests\n\nRun the test suite using pytest:\n$ pytest\n\n## Pull Requests\n\n1. Create a branch from main: git checkout -b my-feature\n2. Write unit tests in tests/\n3. Update CHANGES.rst for bug fixes or features.""",
    "CODE_OF_CONDUCT.md": """# Contributor Covenant Code of Conduct\n\nWe pledge to make participation a harassment-free experience for everyone regardless of identity. Be respectful of differing opinions and experiences. Sexualized language or harassment is unacceptable.""",
    "docs/testing.md": """# Testing Flask\n\nFlask uses pytest for unit testing.\nTo run the test suite:\n$ pytest\nTo check coverage:\n$ pytest --cov=flask""",
    "LICENSE.md": """# License\n\nFlask is licensed under the BSD 3-Clause License.\nCopyright 2010 Pallets."""
}

print(f"Loaded {len(SAMPLE_DOCS)} documentation files.")

## Step 3: Chunking & Local Sentence-Transformers Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")

chunks = []
for path, content in SAMPLE_DOCS.items():
    lines = content.splitlines()
    words = content.split()
    step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for i in range(0, len(words), step):
        chunk_text = " ".join(words[i : i + CHUNK_SIZE])
        chunks.append({
            "file_path": path,
            "content": chunk_text,
            "start_line": 1,
            "end_line": len(lines)
        })

# Compute unit-normalized embeddings
chunk_texts = [f"[{c['file_path']}] {c['content']}" for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)

print(f"Created {len(chunks)} chunks and computed embeddings on CPU.")

## Step 4: Semantic Retrieval with Cosine Similarity

In [ ]:
def retrieve(query, top_k=TOP_K):
    """Retrieves top-k closest chunks using dot product over normalized vectors."""
    q_vec = embed_model.encode(query, normalize_embeddings=True)
    sims = np.dot(chunk_embeddings, q_vec)
    top_idx = np.argsort(sims)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_idx, start=1):
        results.append({
            "rank": rank,
            "chunk": chunks[idx],
            "similarity": float(sims[idx])
        })
    return results

print("Retriever initialized.")

## Step 5: Answer Generation with Gemma 4 & Safe Refusal

In [ ]:
def get_api_key():
    try:
        from google.colab import userdata
        return userdata.get("GOOGLE_API_KEY")
    except Exception:
        return os.environ.get("GOOGLE_API_KEY")

def generate_answer(query):
    passages = retrieve(query, top_k=TOP_K)
    top_score = passages[0]["similarity"]

    # Safe refusal guardrail
    if top_score < MIN_RELEVANCE_THRESHOLD:
        return (
            "I could not verify that from this repository's documentation. "
            "The indexed documentation does not contain sufficient evidence to answer this question reliably.",
            passages
        )

    api_key = get_api_key()
    context_parts = []
    for p in passages:
        c = p["chunk"]
        context_parts.append(f"[Passage {p['rank']}: {c['file_path']} (Lines {c['start_line']}-{c['end_line']})]\n{c['content']}")
    context_text = "\n\n".join(context_parts)

    if not api_key:
        # Demonstration mode answer
        top_c = passages[0]["chunk"]
        demo = f"{top_c['content'][:180]}... (Refer to {top_c['file_path']}). [Demonstration Mode: Configure GOOGLE_API_KEY in Colab Secrets for live Gemma 4 generation]"
        return demo, passages

    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model=GEMMA_MODEL,
            contents=f"Context passages:\n{context_text}\n\nQuestion: {query}\nAnswer:",
            config=types.GenerateContentConfig(
                system_instruction=ANSWER_RULES,
                temperature=TEMPERATURE
            )
        )
        return response.text, passages
    except Exception as e:
        return f"Error with Gemma 4 API ({e}).", passages

print("Gemma 4 generation pipeline ready.")

## Step 6: 'Where Do I Start?' Onboarding Checklist

In [ ]:
def generate_checklist():
    steps = [
        ("1. Review Code of Conduct", "code of conduct community standards"),
        ("2. Set Up Virtual Environment", "python3 -m venv virtualenv install pip"),
        ("3. Run Automated Tests", "run tests pytest test suite"),
        ("4. Pick an Issue", "finding good first issue contribute"),
        ("5. Submit Pull Request", "pull request branch changes changelog"),
    ]
    checklist = []
    for title, q in steps:
        ret = retrieve(q, top_k=1)
        if ret and ret[0]["similarity"] >= 0.35:
            c = ret[0]["chunk"]
            checklist.append(f"✓ {title} — Documented in `{c['file_path']}`: {c['content'][:120]}...")
        else:
            checklist.append(f"✗ {title} — Not covered in repository documentation")
    return "\n".join(checklist)

print(generate_checklist())

## Step 7: 16-Question Evaluation Benchmark

In [ ]:
BENCHMARK = [
    ("How do I set up a local development sandbox?", True),
    ("What Python version is required?", True),
    ("How do I install in editable development mode?", True),
    ("How do I run tests with pytest?", True),
    ("Are there pre-commit hooks configured?", True),
    ("What are the branch naming and PR rules?", True),
    ("Is there a Code of Conduct?", True),
    ("What is the project license?", True),
    ("How do I check test coverage?", True),
    ("Where is the changelog updated?", True),
    ("What is the CEO's personal phone number?", False),
    ("How do I deploy on Kubernetes in AWS GovCloud?", False),
    ("What was quarterly revenue in 2024?", False),
    ("Can I write PRs in Rust?", False),
    ("What is the root database password?", False),
    ("Who won the UEFA Champions League in 2023?", False),
]

def run_benchmark():
    hits, refusals = 0, 0
    for q, answerable in BENCHMARK:
        ans, passg = generate_answer(q)
        is_refusal = "could not" in ans.lower() or "not found" in ans.lower()
        if is_refusal == (not answerable):
            refusals += 1
        if answerable and passg[0]["similarity"] >= 0.32:
            hits += 1
    print(f"Retrieval Hit Rate: {hits}/10 ({hits * 10}%)")
    print(f"Safe Refusal Rate: {refusals}/16 ({(refusals / 16) * 100:.1f}%)")

run_benchmark()

## Step 8: Interactive Gradio Application

In [ ]:
import gradio as gr

def rag_chat(user_question):
    answer, passages = generate_answer(user_question)
    evidence_md = "### Retrieved Passages & Vector Relevance\n"
    for p in passages:
        c = p["chunk"]
        evidence_md += f"- **[{c['file_path']} #L{c['start_line']}-{c['end_line']}]** (sim: `{p['similarity']:.4f}`)\n> {c['content'][:180]}...\n\n"
    return answer, evidence_md

with gr.Blocks(title="Orynth — Contributor Guide") as demo:
    gr.Markdown("# orynth\n*Evidence-grounded open source contributor guide powered by Gemma 4*")
    with gr.Row():
        with gr.Column(scale=2):
            inp = gr.Textbox(label="Ask about repository setup, contribution rules, tests, or PRs:", placeholder="e.g. How do I set up a local development environment?")
            btn = gr.Button("Inquire", variant="primary")
            out_ans = gr.Markdown(label="Grounded Answer")
        with gr.Column(scale=1):
            out_passages = gr.Markdown(label="Retrieved Passages")
    btn.click(rag_chat, inputs=[inp], outputs=[out_ans, out_passages])

demo.launch(share=True)